# Clustering IO & Apply

Cluster on a subset of your data, save the clustering, and apply it
to the full dataset later. This is the typical workflow when you want
to determine clusters from renewable generation profiles and then
apply the same clustering to all variables including demand.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook"

da = sample_energy_data(n_days=30).sel(region="north", scenario="low")
print(f"Shape: {dict(da.sizes)}")

## Cluster on renewable generation only

In [ ]:
# Cluster using only solar and wind profiles
da_renewables = da.sel(variable=["solar", "wind"])

result_renewables = tsam_xarray.aggregate(
    da_renewables,
    time_dim="time",
    cluster_dim="variable",
    n_clusters=4,
)

result_renewables.clustering.to_json("clustering.json")
print(f"Clustered on: {list(da_renewables.coords['variable'].values)}")
print(f"Saved {result_renewables.n_clusters} clusters")
result_renewables.typical_periods.plotly.line(
    x="timestep",
    color="cluster",
    facet_col="variable",
    title="Clustering based on renewables only",
)

## Apply to all variables

Load the clustering and apply it to the full dataset including demand.
The cluster assignments (which days go together) stay the same — only
the representatives are recomputed for the new variables.

In [ ]:
clustering = tsam_xarray.load_clustering("clustering.json")

# Apply to ALL variables (solar, wind, AND demand)
result_all = clustering.apply(da)
print(f"Variables: {list(result_all.typical_periods.coords['variable'].values)}")
result_all.typical_periods.plotly.line(
    x="timestep",
    color="cluster",
    facet_col="variable",
    title="Same clustering applied to all variables",
)

## Compare accuracy

In [ ]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "renewables only": result_renewables.accuracy.rmse.to_series(),
        "all variables (applied)": result_all.accuracy.rmse.to_series(),
    }
)
comparison